# 08b – Kiến trúc riêng trên chuỗi đặc trưng theo thời gian của XLS-R (hướng C, lần 2)

**Khác lần 1:** giữ đủ 1024 chiều (không giảm PCA), train trên cả câu 12 giây (không cắt đoạn ngẫu nhiên), 60 epoch, chỉ chạy 3 kiến trúc `mean`, `conv_asp`, `transformer`.
Ở lần 1, `mean` chỉ đạt 49,0%, trong khi head LP trên cùng đặc trưng đạt 53,2%. Pipeline đánh mất khoảng 4 điểm, nên phép so sánh kiến trúc chưa công bằng.
Kết quả lần 1 vẫn được giữ nguyên trong `results/frame_runs/`.

Các notebook trước **lấy trung bình theo thời gian** rồi mới đưa vào MLP. Cách đó bỏ mất thông tin về *diễn biến* âm thanh theo thời gian (đường nét thanh điệu, cách chuyển âm), vốn là chỗ phương ngữ khác nhau rõ nhất.
Notebook này giữ nguyên **chuỗi frame** và so sánh 5 kiến trúc tự xây:

| Kiến trúc | Ý tưởng |
|---|---|
| `mean` | Trung bình theo thời gian → MLP. Tương đương mô hình cũ, dùng làm **đối chứng** |
| `stats` | Trung bình + độ lệch chuẩn theo thời gian → MLP |
| `asp` | Attentive statistics pooling (Okabe et al., 2018): học frame nào quan trọng |
| `conv_asp` | ECAPA-lite: các khối Conv1D giãn nở + Squeeze-Excitation (Desplanques et al., 2020) để bắt mẫu cục bộ theo thời gian, sau đó ASP |
| `transformer` | 2 lớp Transformer encoder trên chuỗi frame, sau đó ASP |

**Quy trình:**
1. Lấy trọng số các tầng từ head LP (13 tầng).
2. Trích chuỗi frame **một lần** (XLS-R đóng băng → weighted-sum các tầng → 25 frame/giây → PCA xuống 512 chiều), đặt toàn bộ lên GPU.
3. Train 5 kiến trúc với LS, mỗi kiến trúc 3 seed. Khi train cắt ngẫu nhiên đoạn 6 giây và che ngẫu nhiên vài khoảng thời gian (time masking).
4. Kiến trúc tốt nhất (theo valid) và `mean` được train thêm với Geo-LS.

- **Runtime:** GPU L4/A100. Trích đặc trưng khoảng 15–20 phút, train khoảng 30–60 phút.
- **Bị ngắt:** chạy lại, các lần chạy đã xong sẽ tự bỏ qua.
- **GPU:** cần **L4 trở lên**, vì đặc trưng chiếm khoảng 11,6 GB VRAM; T4 16 GB sẽ không đủ.
- **Kết quả:** `results/frame_report_1024d_12s.json` và `fig_arch_compare_1024d_12s.png`. Gửi cả hai cho Claude.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL = "province_code"
XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"
KEEP_LAYERS = 12
EVAL_SEC = 12.0          # độ dài tối đa mỗi câu (giây)
POOL_T = 2               # gộp 2 frame → 25 frame/giây
PCA_DIM = 1024           # = số chiều gốc của XLS-R → không làm mất thông tin
PCA_FIT_UTTS = 1500
EXTRACT_BATCH = 16
ARCHS = ["mean", "conv_asp", "transformer"]
SEEDS = [0, 1, 2]
EPOCHS, BATCH, LR, PATIENCE = 60, 128, 1e-3, 10
CROP_SEC = 12.0          # = EVAL_SEC → train trên cả câu, giống cách tính đặc trưng LP
WORKERS = 4

In [ ]:
import os, io, glob, json, time, math, shutil, warnings, multiprocessing as mp
import numpy as np, pandas as pd, pyarrow.parquet as pq, soundfile as sf
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, top_k_accuracy_score
warnings.filterwarnings("ignore")
T0 = time.time()
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
LOCAL = "/content" if os.path.isdir("/content") else os.path.abspath("_work")
LOCAL_DATA, FEAT_DIR, RES_DIR = f"{LOCAL}/vimd16k", f"{DRIVE_ROOT}/features", f"{DRIVE_ROOT}/results"
TAG = f"{PCA_DIM}d_{int(CROP_SEC)}s"
RUN_DIR = f"{RES_DIR}/frame_runs_{TAG}"
for d in (LOCAL_DATA, RUN_DIR):
    os.makedirs(d, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == "cuda" else None
autocast = lambda: torch.autocast(device_type=device.type, dtype=AMP or torch.bfloat16, enabled=AMP is not None)
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "", "| AMP:", AMP)

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
prov = pd.read_csv(f"{DRIVE_ROOT}/provinces.csv"); prov[PROVINCE_COL] = prov[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
tr, va, te = [(meta["split"].values == s) & use for s in ("train", "valid", "test")]
pidx = prov.set_index(PROVINCE_COL)
NAMES = [pidx.loc[c, "province_name"] for c in CLASSES]
region_of = np.array([pidx.loc[c, "region"] for c in CLASSES])
yt = y[te]
N = len(meta)

COORD = {
 "CaoBang": (22.666, 106.258), "LangSon": (21.853, 106.761), "QuangNinh": (20.951, 107.080), "HaiPhong": (20.845, 106.688),
 "ThaiBinh": (20.446, 106.336), "NamDinh": (20.420, 106.168), "PhuTho": (21.322, 105.402), "ThaiNguyen": (21.594, 105.848),
 "YenBai": (21.705, 104.875), "TuyenQuang": (21.823, 105.214), "HaGiang": (22.823, 104.984), "LaoCai": (22.486, 103.971),
 "LaiChau": (22.396, 103.458), "SonLa": (21.326, 103.917), "DienBien": (21.386, 103.023), "HoaBinh": (20.817, 105.338),
 "HaNoi": (21.028, 105.854), "HaiDuong": (20.940, 106.333), "NinhBinh": (20.251, 105.975), "VinhPhuc": (21.310, 105.597),
 "HungYen": (20.646, 106.051), "HaNam": (20.541, 105.914), "BacKan": (22.147, 105.835), "BacGiang": (21.273, 106.195),
 "BacNinh": (21.186, 106.076), "ThanhHoa": (19.807, 105.776), "NgheAn": (18.679, 105.682), "HaTinh": (18.343, 105.906),
 "QuangBinh": (17.468, 106.622), "QuangTri": (16.816, 107.100), "ThuaThienHue": (16.464, 107.586), "DaNang": (16.054, 108.202),
 "QuangNam": (15.573, 108.474), "QuangNgai": (15.120, 108.792), "BinhDinh": (13.776, 109.224), "PhuYen": (13.096, 109.309),
 "KhanhHoa": (12.239, 109.197), "NinhThuan": (11.565, 108.988), "BinhThuan": (10.928, 108.102), "KonTum": (14.350, 108.000),
 "GiaLai": (13.983, 108.000), "DakLak": (12.667, 108.038), "DakNong": (12.004, 107.690), "LamDong": (11.940, 108.458),
 "HoChiMinh": (10.776, 106.701), "DongNai": (10.957, 106.843), "BinhDuong": (10.980, 106.651), "BaRiaVungTau": (10.496, 107.169),
 "TayNinh": (11.310, 106.098), "BinhPhuoc": (11.535, 106.883), "LongAn": (10.535, 106.413), "TienGiang": (10.360, 106.360),
 "BenTre": (10.241, 106.376), "VinhLong": (10.254, 105.972), "TraVinh": (9.935, 106.345), "CanTho": (10.045, 105.747),
 "HauGiang": (9.784, 105.470), "SocTrang": (9.603, 105.980), "BacLieu": (9.294, 105.727), "CaMau": (9.176, 105.150),
 "DongThap": (10.460, 105.633), "AnGiang": (10.386, 105.435), "KienGiang": (10.012, 105.081),
}
LAT = np.array([COORD[n][0] for n in NAMES]); LON = np.array([COORD[n][1] for n in NAMES])
p_ = np.pi / 180
DIST = 2 * 6371 * np.arcsin(np.sqrt(np.sin((LAT[None] - LAT[:, None]) * p_ / 2) ** 2 + np.cos(LAT[:, None] * p_)
                                    * np.cos(LAT[None] * p_) * np.sin((LON[None] - LON[:, None]) * p_ / 2) ** 2))
EYE = torch.eye(C, device=device); DIST_t = torch.tensor(DIST, dtype=torch.float32, device=device)

def target_matrix(kind, p):
    if kind == "ce":
        return EYE
    if kind == "ls":
        return (1 - p) * EYE + p / C
    if kind == "geols":
        eps, tau = p
        nb = torch.exp(-DIST_t / tau) * (1 - EYE)
        return (1 - eps) * EYE + eps * nb / nb.sum(1, keepdim=True)
    raise ValueError(kind)

def metrics(P, yy):
    pred = P.argmax(1); d = DIST[yy, pred]
    return dict(acc=(pred == yy).mean(), macro_f1=f1_score(yy, pred, average="macro"),
                top5=top_k_accuracy_score(yy, P, k=min(5, C - 1), labels=np.arange(C)),
                region_acc=(region_of[pred] == region_of[yy]).mean(), mean_err_km=d.mean(),
                within_100km=(d <= 100).mean(), expected_km=(P * DIST[yy]).sum(1).mean())

rep = f"{RES_DIR}/geo_ls_report_{KEEP_LAYERS + 1}layers.json"
rep = rep if os.path.exists(rep) else f"{RES_DIR}/geo_ls_report.json"
ch = json.load(open(rep))["chosen"]
LS = ("ls", float(ch["ls"]))
GEO = ("geols", tuple(float(v) for v in ch["geols"].strip("()").split(",")))
print("Cấu hình hàm mục tiêu lấy từ", os.path.basename(rep), "→", LS, GEO)

## 1. Trọng số các tầng: lấy từ head LP train trên đặc trưng đã cache (13 tầng)

In [ ]:
fdir = sorted(glob.glob(f"{FEAT_DIR}/xlsr_*"))[0]
GROUPS = {s: (g.index.values, g["row"].values) for s, g in meta.groupby("shard")}
first = np.load(f"{fdir}/{next(iter(GROUPS))}.npy")
XL = np.zeros((N, KEEP_LAYERS + 1, first.shape[2]), np.float16)
for s, (idx, _) in GROUPS.items():
    XL[idx] = np.load(f"{fdir}/{s}.npy")[:, :KEEP_LAYERS + 1]
Xp = torch.tensor(XL, device=device); del XL
Xtr_ = Xp[torch.tensor(tr, device=device)].float()
MU, SD = Xtr_.mean(0), Xtr_.std(0) + 1e-5                       # (L, D)
torch.manual_seed(0)
w_raw = nn.Parameter(torch.zeros(KEEP_LAYERS + 1, device=device))
mlp = nn.Sequential(nn.Linear(Xp.shape[2], 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C)).to(device)
opt = torch.optim.AdamW([w_raw, *mlp.parameters()], lr=1e-3, weight_decay=1e-3)
ytr_ = torch.tensor(y[tr], device=device); Q = target_matrix(*LS)
lp_fwd = lambda X: mlp((((X.float() - MU) / SD) * w_raw.softmax(0)[None, :, None]).sum(1))
for ep in range(30):
    perm = torch.randperm(len(Xtr_), device=device)
    for i in range(0, len(perm), 256):
        b = perm[i:i + 256]
        loss = -(Q[ytr_[b]] * F.log_softmax(lp_fwd(Xtr_[b]), -1)).sum(1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
LAYER_W = w_raw.softmax(0).detach()
mlp.eval()
with torch.no_grad():
    lp_va = (lp_fwd(Xp[torch.tensor(va, device=device)]).argmax(1).cpu().numpy() == y[va]).mean()
print("Trọng số tầng:", LAYER_W.cpu().numpy().round(3), f"| LP valid acc {lp_va:.4f}")
del Xp, Xtr_, mlp, opt; torch.cuda.empty_cache()

## 2. Waveform memmap (giống notebook 05, 06)

In [ ]:
for f in sorted(glob.glob(f"{DRIVE_ROOT}/data16k/*.parquet")):
    dst = f"{LOCAL_DATA}/{os.path.basename(f)}"
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f):
        shutil.copy(f, dst)
MAXS = int(EVAL_SEC * 16000)
WAV_PATH, LEN_PATH = f"{LOCAL}/vimd_wave_{int(EVAL_SEC)}s.int16", f"{LOCAL}/vimd_wave_{int(EVAL_SEC)}s.len.npy"

def fill_shard(shard):
    idx, rows = GROUPS[shard]
    mm = np.memmap(WAV_PATH, dtype=np.int16, mode="r+", shape=(N, MAXS))
    col = pq.read_table(f"{LOCAL_DATA}/{shard}", columns=["audio"]).column(0)
    out = []
    for mi, r in zip(idx, rows):
        w = sf.read(io.BytesIO(col[int(r)].as_py()), dtype="int16")[0][:MAXS]
        mm[mi, :len(w)] = w; out.append((mi, len(w)))
    mm.flush()
    return out

if not os.path.exists(LEN_PATH):
    t = time.time()
    np.memmap(WAV_PATH, dtype=np.int16, mode="w+", shape=(N, MAXS)).flush()
    LENS = np.zeros(N, np.int64)
    with mp.get_context("fork").Pool(os.cpu_count()) as pool:
        for res in pool.imap_unordered(fill_shard, list(GROUPS)):
            for mi, n in res:
                LENS[mi] = n
    np.save(LEN_PATH, LENS)
    print(f"Tạo memmap xong sau {time.time() - t:.0f}s")
LENS = np.load(LEN_PATH)
WAV = np.memmap(WAV_PATH, dtype=np.int16, mode="r", shape=(N, MAXS))

## 3. Trích chuỗi frame (một lần): XLS-R → weighted-sum → 25 frame/giây → PCA 512 chiều

In [ ]:
from transformers import AutoModel
TMAX = int(EVAL_SEC * 50) // POOL_T
FE_PATH, FL_PATH = f"{LOCAL}/frames_{KEEP_LAYERS}L_{PCA_DIM}d.f16", f"{LOCAL}/frames_{KEEP_LAYERS}L_{PCA_DIM}d.len.npy"

def load_encoder():
    enc = AutoModel.from_pretrained(XLSR_MODEL, num_hidden_layers=KEEP_LAYERS, layerdrop=0.0)
    if getattr(enc.config, "do_stable_layer_norm", False):
        enc.encoder.layer_norm = nn.Identity()
    return enc.to(device).eval()

@torch.no_grad()
def frame_batch(enc, idx):
    ws = [WAV[i, :LENS[i]].astype(np.float32) / 32768.0 for i in idx]
    L = max(len(w) for w in ws)
    x = np.zeros((len(ws), L), np.float32); m = np.zeros((len(ws), L), np.int64)
    for j, w in enumerate(ws):
        w = (w - w.mean()) / (w.std() + 1e-7); x[j, :len(w)] = w; m[j, :len(w)] = 1
    x, m = torch.from_numpy(x).to(device), torch.from_numpy(m).to(device)
    with autocast():
        hs = enc(x, attention_mask=m, output_hidden_states=True).hidden_states[:KEEP_LAYERS + 1]
    z = sum(LAYER_W[l] * (hs[l].float() - MU[l]) / SD[l] for l in range(KEEP_LAYERS + 1))      # B, T, D
    flen = enc._get_feat_extract_output_lengths(m.sum(1)).long()
    T2 = z.shape[1] // POOL_T
    z = z[:, :T2 * POOL_T].reshape(z.shape[0], T2, POOL_T, -1).mean(2)
    return z, torch.clamp(flen // POOL_T, 1, T2)

if not os.path.exists(FL_PATH):
    enc = load_encoder(); t = time.time()
    rng = np.random.default_rng(0)
    fit_idx = rng.choice(np.where(tr)[0], min(PCA_FIT_UTTS, tr.sum()), replace=False)
    S = []
    for i in range(0, len(fit_idx), EXTRACT_BATCH):
        z, fl = frame_batch(enc, fit_idx[i:i + EXTRACT_BATCH])
        for j in range(len(z)):
            fr = z[j, :fl[j]]; S.append(fr[torch.randperm(len(fr), device=device)[:50]])
    S = torch.cat(S).float(); PCA_MEAN = S.mean(0)
    evals, evecs = torch.linalg.eigh(((S - PCA_MEAN).T @ (S - PCA_MEAN)) / len(S))
    P = evecs[:, -PCA_DIM:].flip(1)
    EXPL = float(evals[-PCA_DIM:].sum() / evals.sum())
    print(f"PCA {S.shape[1]}→{PCA_DIM}: giữ {EXPL:.1%} phương sai | {time.time() - t:.0f}s")
    FE = np.memmap(FE_PATH, dtype=np.float16, mode="w+", shape=(N, TMAX, PCA_DIM))
    FL = np.zeros(N, np.int64)
    order = np.argsort(-LENS)
    for k in range(0, N, EXTRACT_BATCH):
        idx = order[k:k + EXTRACT_BATCH]
        z, fl = frame_batch(enc, idx)
        z = ((z - PCA_MEAN) @ P).half().cpu().numpy()
        T2 = min(z.shape[1], TMAX)
        FE[idx, :T2] = z[:, :T2]; FL[idx] = np.minimum(fl.cpu().numpy(), T2)
        if (k // EXTRACT_BATCH) % 100 == 0:
            el = time.time() - t
            print(f"  {k}/{N} câu | {el / 60:.1f} phút", flush=True)
    FE.flush(); np.save(FL_PATH, FL)
    json.dump(dict(pca_explained=EXPL), open(f"{LOCAL}/frames_info.json", "w"))
    del enc; torch.cuda.empty_cache()
    print(f"Trích xong sau {(time.time() - t) / 60:.1f} phút")
FL = np.load(FL_PATH)
INFO = json.load(open(f"{LOCAL}/frames_info.json"))
FEg = torch.empty((N, TMAX, PCA_DIM), dtype=torch.float16, device=device)
FE = np.memmap(FE_PATH, dtype=np.float16, mode="r", shape=(N, TMAX, PCA_DIM))
for i in range(0, N, 2048):
    FEg[i:i + 2048] = torch.from_numpy(np.ascontiguousarray(FE[i:i + 2048])).to(device)
FLg = torch.tensor(FL, device=device)
print(f"Chuỗi frame trên GPU: {tuple(FEg.shape)} ({FEg.numel() * 2 / 1e9:.1f} GB) | độ dài trung vị {np.median(FL) * POOL_T / 50:.1f}s")

## 4. Năm kiến trúc

In [ ]:
def masked_mean(x, m):
    m = m.float()[..., None]
    return (x * m).sum(1) / m.sum(1).clamp(min=1)

def masked_std(x, m, mu):
    m = m.float()[..., None]
    return (((x - mu[:, None]) ** 2 * m).sum(1) / m.sum(1).clamp(min=1)).clamp(min=1e-6).sqrt()

class ASP(nn.Module):
    """Attentive statistics pooling (Okabe et al., 2018)."""
    def __init__(s, d, h=128):
        super().__init__(); s.att = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Linear(h, 1))
    def forward(s, x, m):
        a = s.att(x).squeeze(-1).float().masked_fill(~m, -1e4).softmax(1)[..., None]
        mu = (a * x).sum(1); sd = ((a * x * x).sum(1) - mu ** 2).clamp(min=1e-6).sqrt()
        return torch.cat([mu, sd], -1)

def head(d_in):
    return nn.Sequential(nn.Linear(d_in, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))

class MeanNet(nn.Module):
    def __init__(s, d):
        super().__init__(); s.h = head(d)
    def forward(s, x, m):
        return s.h(masked_mean(x, m))

class StatsNet(nn.Module):
    def __init__(s, d):
        super().__init__(); s.h = head(2 * d)
    def forward(s, x, m):
        mu = masked_mean(x, m); return s.h(torch.cat([mu, masked_std(x, m, mu)], -1))

class ASPNet(nn.Module):
    def __init__(s, d):
        super().__init__(); s.pool = ASP(d); s.h = head(2 * d)
    def forward(s, x, m):
        return s.h(s.pool(x, m))

class SERes(nn.Module):
    """Khối Conv1D giãn nở + Squeeze-Excitation có residual (ECAPA-lite)."""
    def __init__(s, c, k, dil):
        super().__init__()
        s.c1 = nn.Conv1d(c, c, k, dilation=dil, padding=dil * (k - 1) // 2); s.b1 = nn.BatchNorm1d(c)
        s.c2 = nn.Conv1d(c, c, 1); s.b2 = nn.BatchNorm1d(c)
        s.se = nn.Sequential(nn.Linear(c, c // 8), nn.ReLU(), nn.Linear(c // 8, c), nn.Sigmoid())
    def forward(s, x, m):                      # x: B, C, T
        mf = m.float()[:, None]
        h = F.relu(s.b1(s.c1(x))) * mf
        h = F.relu(s.b2(s.c2(h))) * mf
        g = s.se((h.sum(2) / mf.sum(2).clamp(min=1)))[..., None]
        return (x + h * g) * mf

class ConvASPNet(nn.Module):
    def __init__(s, d, c=256):
        super().__init__(); s.inp = nn.Linear(d, c)
        s.blocks = nn.ModuleList([SERes(c, 5, dl) for dl in (1, 2, 3)])
        s.agg = nn.Conv1d(3 * c, 2 * c, 1); s.pool = ASP(2 * c); s.h = head(4 * c)
    def forward(s, x, m):
        h = s.inp(x).transpose(1, 2); outs = []
        for b in s.blocks:
            h = b(h, m); outs.append(h)
        h = F.relu(s.agg(torch.cat(outs, 1))).transpose(1, 2)
        return s.h(s.pool(h, m))

class TransNet(nn.Module):
    def __init__(s, d, c=256, layers=2):
        super().__init__(); s.inp = nn.Linear(d, c)
        pe = torch.zeros(1024, c); pos = torch.arange(1024)[:, None]
        div = torch.exp(torch.arange(0, c, 2) * (-math.log(10000.0) / c))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        s.register_buffer("pe", pe)
        s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(c, 4, 2 * c, dropout=0.1, batch_first=True,
                                                                 norm_first=True), layers)
        s.pool = ASP(c); s.h = head(2 * c)
    def forward(s, x, m):
        h = s.inp(x) + s.pe[:x.shape[1]][None]
        h = s.enc(h, src_key_padding_mask=~m)
        return s.h(s.pool(h, m))

BUILD = dict(mean=MeanNet, stats=StatsNet, asp=ASPNet, conv_asp=ConvASPNet, transformer=TransNet)

## 5. Huấn luyện (toàn bộ dữ liệu nằm sẵn trên GPU)

In [ ]:
CROP = int(CROP_SEC * 50) // POOL_T
ar = torch.arange(TMAX, device=device)

def get_batch(idx, train):
    idx = torch.as_tensor(idx, device=device); L = FLg[idx]
    if train:
        st = (torch.rand(len(idx), device=device) * (L - CROP).clamp(min=0).float()).long()
        pos = (st[:, None] + ar[None, :CROP]).clamp(max=TMAX - 1)
        x = FEg[idx[:, None], pos].float(); m = ar[None, :CROP] < (L - st)[:, None]
        for _ in range(2):                                         # time masking
            w = torch.randint(0, 6, (len(idx), 1), device=device)
            s0 = (torch.rand(len(idx), 1, device=device) * CROP).long()
            m = m & ~((ar[None, :CROP] >= s0) & (ar[None, :CROP] < s0 + w))
        m[:, 0] |= ~m.any(1)
        return x, m
    T = int(L.max()); return FEg[idx, :T].float(), ar[None, :T] < L[:, None]

@torch.no_grad()
def predict(model, mask, bs=256):
    model.eval(); idx = np.where(mask)[0]; out = []
    for i in range(0, len(idx), bs):
        x, m = get_batch(idx[i:i + bs], False)
        with autocast():
            out.append(model(x, m).float().softmax(-1).cpu())
    return torch.cat(out).numpy()

def train_run(arch, loss, seed):
    name = f"{arch}_{loss[0]}_{seed}"
    if os.path.exists(f"{RUN_DIR}/{name}.json"):
        return json.load(open(f"{RUN_DIR}/{name}.json"))
    torch.manual_seed(seed); np.random.seed(seed)
    model = BUILD[arch](PCA_DIM).to(device)
    lr = LR / 2 if arch == "transformer" else LR
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    tr_idx = np.where(tr)[0]; spe = len(tr_idx) // BATCH
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EPOCHS * spe, pct_start=0.05)
    Q = target_matrix(*loss); best, state, bad, t = -1, None, 0, time.time()
    for ep in range(EPOCHS):
        model.train(); perm = np.random.permutation(tr_idx)
        for i in range(spe):
            b = perm[i * BATCH:(i + 1) * BATCH]; x, m = get_batch(b, True)
            with autocast():
                logits = model(x, m)
            l = -(Q[torch.as_tensor(y[b], device=device)] * F.log_softmax(logits.float(), -1)).sum(1).mean()
            opt.zero_grad(set_to_none=True); l.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); sched.step()
        acc = (predict(model, va).argmax(1) == y[va]).mean()
        if acc > best:
            best, state, bad = acc, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(state)
    Pt = predict(model, te); np.save(f"{RUN_DIR}/{name}_test_probs.npy", Pt)
    r = dict(arch=arch, loss=loss[0], param=str(loss[1]), seed=seed, epochs=ep + 1, minutes=(time.time() - t) / 60,
             params=sum(p.numel() for p in model.parameters()),
             **{f"valid_{k}": float(v) for k, v in metrics(predict(model, va), y[va]).items()},
             **{f"test_{k}": float(v) for k, v in metrics(Pt, yt).items()})
    json.dump(r, open(f"{RUN_DIR}/{name}.json", "w"), indent=1)
    del model, opt; torch.cuda.empty_cache()
    return r

RUNS = []
for arch in ARCHS:
    for s in SEEDS:
        r = train_run(arch, LS, s); RUNS.append(r)
        print(f"{arch:11s} LS  seed {s} | valid {r['valid_acc']:.4f} | test {r['test_acc']:.4f} | "
              f"err {r['test_mean_err_km']:.0f} km | {r['params'] / 1e6:.2f}M tham số | {r['minutes']:.1f} phút", flush=True)

runs = pd.DataFrame(RUNS)
best_arch = runs.groupby("arch")["valid_acc"].mean().idxmax()
print(f"\nKiến trúc tốt nhất theo valid: {best_arch}")
for arch in dict.fromkeys([best_arch, "mean"]):
    for s in SEEDS:
        r = train_run(arch, GEO, s); RUNS.append(r)
        print(f"{arch:11s} GEO seed {s} | valid {r['valid_acc']:.4f} | test {r['test_acc']:.4f} | err {r['test_mean_err_km']:.0f} km", flush=True)
runs = pd.DataFrame(RUNS)

## 6. Tổng hợp, so sánh cặp và hình

In [ ]:
cols = ["acc", "macro_f1", "top5", "region_acc", "mean_err_km", "within_100km"]
g = runs.groupby(["arch", "loss"])
TABLE = pd.concat([g[[f"test_{c}" for c in cols]].mean().round(4).add_suffix("_mean"),
                   g[[f"test_{c}" for c in cols]].std().round(4).add_suffix("_std"),
                   g["valid_acc"].mean().round(4), g["params"].first()], axis=1).reset_index()
print(TABLE[["arch", "loss", "valid_acc", "test_acc_mean", "test_acc_std", "test_mean_err_km_mean", "params"]].to_string(index=False))

ens = lambda a, l: np.mean([np.load(f"{RUN_DIR}/{a}_{l}_{s}_test_probs.npy") for s in SEEDS], 0).argmax(1)
B = np.random.default_rng(0).integers(0, len(yt), (2000, len(yt)))
def paired(a, b):
    dacc = (a == yt).astype(float) - (b == yt); dkm = DIST[yt, a] - DIST[yt, b]
    return dict(acc_diff=round(float(dacc.mean()), 4), acc_ci95=np.percentile(dacc[B].mean(1), [2.5, 97.5]).round(4).tolist(),
                km_diff=round(float(dkm.mean()), 1), km_ci95=np.percentile(dkm[B].mean(1), [2.5, 97.5]).round(1).tolist())
COMPARE = {f"{a}(ls) - mean(ls)": paired(ens(a, "ls"), ens("mean", "ls")) for a in ARCHS if a != "mean"}
COMPARE[f"{best_arch}(geols) - {best_arch}(ls)"] = paired(ens(best_arch, "geols"), ens(best_arch, "ls"))
if best_arch != "mean":
    COMPARE["mean(geols) - mean(ls)"] = paired(ens("mean", "geols"), ens("mean", "ls"))
print(json.dumps(COMPARE, indent=1))

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                     "axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e"})
ls_tab = TABLE[TABLE.loss == "ls"].set_index("arch").loc[ARCHS]
fig, ax = plt.subplots(figsize=(5.4, 2.8))
yy = np.arange(len(ARCHS))[::-1]
mu_, sd_ = ls_tab["test_acc_mean"].values, ls_tab["test_acc_std"].fillna(0).values
ax.errorbar(mu_, yy, xerr=sd_, fmt="o", color="#2a78d6", ecolor="#52514e", elinewidth=1, capsize=3, markersize=7)
for yv, m_, s_ in zip(yy, mu_, sd_):
    ax.text(m_ + s_ + 0.003, yv, f"{m_:.1%}", va="center", fontsize=8, color="#0b0b0b")
ax.set_yticks(yy, ARCHS); ax.set_xlabel(f"Accuracy test (trung bình ± độ lệch chuẩn, {len(SEEDS)} seed)")
ax.set_xlim((mu_ - sd_).min() - 0.02, (mu_ + sd_).max() + 0.04)
ax.set_title("So sánh kiến trúc trên chuỗi frame XLS-R (loss LS)", fontsize=9, loc="left")
ax.grid(axis="x", color="#e6e5e0", linewidth=0.6); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(f"{RES_DIR}/fig_arch_compare_{TAG}.png"); plt.show()

report = dict(table=TABLE.to_dict(orient="records"), compare=COMPARE, best_arch=best_arch,
              layer_weights=LAYER_W.cpu().numpy().round(4).tolist(), lp_valid_acc=float(lp_va),
              pca_explained=INFO["pca_explained"], losses=dict(ls=str(LS[1]), geols=str(GEO[1])),
              all_runs=RUNS, config=dict(keep_layers=KEEP_LAYERS, pca_dim=PCA_DIM, fps=50 // POOL_T, crop_sec=CROP_SEC,
                                         epochs=EPOCHS, batch=BATCH, lr=LR),
              gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
              total_minutes=round((time.time() - T0) / 60, 1))
json.dump(report, open(f"{RES_DIR}/frame_report_{TAG}.json", "w"), ensure_ascii=False, indent=1, default=float)
print(f"Đã lưu results/frame_report_{TAG}.json + fig_arch_compare_{TAG}.png – gửi cho Claude.")